# treinamento do modelos BERT via HuggingFace

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

2025-05-25 14:51:26.456001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748195486.467436    7688 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748195486.470653    7688 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748195486.480510    7688 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748195486.480529    7688 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748195486.480530    7688 computation_placer.cc:177] computation placer alr

In [ ]:
df = pd.read_csv('../data/processed/olist_reviews_final.csv')
print(df.shape)
df.head()

(20840, 5)


,review_id,review_score,review_comment_message,clean_review,sentiment
0,14c3c20c1d367f82ce9800a9a4575bab,5,"GOSTARIA DE RECEBER 1 COPIA DA NF, DEVE TER EX...",gostaria receber copia nf deve ter extraviado,1
1,5142952a245dd6d9675db172d8c17a03,1,Comprei o relogio para o dia das mães e até ho...,comprei relogio dia maes ate hoje nao chegou,0
2,1b0193dec8f270afea83302dc238f7e6,1,Efetuei a compra e pagamento de duas unidades ...,efetuei compra pagamento duas unidades recebi ...,0
3,6985025ef4acf302504a1446012032a4,1,Recebi o produto errado. Solicitei a troca,recebi produto errado solicitei troca,0
4,acf390bcd82cfecbc91b8b6a1b0f45e8,5,O produto é otimo e correu tudo bem.,produto otimo correu tudo bem,1


# criando dataset huggingface

In [3]:
sub = df[['clean_review','sentiment']].rename(
    columns={'clean_review':'text','sentiment':'labels'}
)
print(sub['labels'].value_counts())

labels
0    10429
1    10411
Name: count, dtype: int64


# definindo treino e teste

In [4]:
X = sub['text']
y = sub['labels']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(X_train)}  Teste: {len(X_test)}")

Treino: 16672  Teste: 4168


# montando dataframe de treino e teste

In [5]:
train_df = pd.DataFrame({'text': X_train, 'labels': y_train}).reset_index(drop=True)
test_df  = pd.DataFrame({'text': X_test,  'labels': y_test}).reset_index(drop=True)
print("train_df:", train_df.shape)
print("test_df :", test_df.shape)

train_df: (16672, 2)
test_df : (4168, 2)


In [6]:
ds_train = Dataset.from_pandas(train_df)
ds_test  = Dataset.from_pandas(test_df)

In [7]:
ds_train = ds_train.class_encode_column('labels')
ds_test  = ds_test.class_encode_column('labels')

Stringifying the column:   0%|          | 0/16672 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/16672 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/4168 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/4168 [00:00<?, ? examples/s]

# tokenização

In [8]:
model_name = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

In [9]:
tokenized_train = ds_train.map(tokenize_fn, batched=True)
tokenized_test  = ds_test.map(tokenize_fn, batched=True)

Map:   0%|          | 0/16672 [00:00<?, ? examples/s]

Map:   0%|          | 0/4168 [00:00<?, ? examples/s]

In [10]:
tokenized_train.set_format(
    type='torch',
    columns=['input_ids','attention_mask','labels']
)
tokenized_test.set_format(
    type='torch',
    columns=['input_ids','attention_mask','labels']
)

# carregando o modelo pre-treinado

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
training_args = TrainingArguments(
    output_dir='models/bert-finetuned',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    save_total_limit=1,
    logging_steps=100,
    logging_dir='logs'
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer
)

/home/vitor/sentiment-analysis/venv/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_7688/3261217663.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# treinamento e avaliação

In [15]:
train_output = trainer.train()
print(train_output)

Epoch,Training Loss,Validation Loss
1,0.227500,0.193520
2,0.138200,0.206955
3,0.121300,0.225701


TrainOutput(global_step=3126, training_loss=0.17145539329209086, metrics={'train_runtime': 662.6142, 'train_samples_per_second': 75.483, 'train_steps_per_second': 4.718, 'total_flos': 3289940636221440.0, 'train_loss': 0.17145539329209086, 'epoch': 3.0})


In [16]:
eval_metrics = trainer.evaluate()
print("Métricas de avaliação:", eval_metrics)

Métricas de avaliação: {'eval_loss': 0.22570116817951202, 'eval_runtime': 13.7419, 'eval_samples_per_second': 303.306, 'eval_steps_per_second': 9.533, 'epoch': 3.0}


In [19]:
# Gerar previsões no conjunto de teste tokenizado
y_pred_output = trainer.predict(tokenized_test)

# Extrair labels verdadeiros e previsões
y_true = y_pred_output.label_ids

y_pred = y_pred_output.predictions.argmax(axis=-1)

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9346    0.9458    0.9402      2086
           1     0.9451    0.9337    0.9394      2082

    accuracy                         0.9398      4168
   macro avg     0.9398    0.9398    0.9398      4168
weighted avg     0.9398    0.9398    0.9398      4168

